# Demo 06 - Cohorts and Clusters

At this point, we're seeing some rather suspicious behavior about Glass and Sons.  They've submitted a large number of invoices just under 1000 USD and a huge number of invoices in the years 2019-2021, but we want to understand better just how unusual their behavior is compared to everyone else.  This is where cohort anlaysis comes into play.

The idea behind cohort anlaysis is to try to understand the behavior of one actor or slice of the data given the remainder.  In our example, we want to understand the data around Glass and Sons with respect to other vendors, as well as one employee's results in the context of other employees.  This makes it easier to observe differences and ask the right sorts of questions.

In [ ]:
import pyodbc
import pandas as pd
import toml

config = toml.load("../config.toml")
server = config["database"]["server"]
database = config["database"]["name"]
username = config["database"]["username"]
password = config["database"]["password"]
conn = pyodbc.connect('DRIVER={ODBC Driver 17 for SQL Server};SERVER='+server+';DATABASE='+database+';UID='+username+';PWD='+password)

In [ ]:
line_items = pd.read_sql("""SELECT
    li.LineItemID,
    c.FirstDayOfMonth,
    c.CalendarYear,
    c.CalendarMonth,
    b.BusID,
    b.AverageRoadConditions,
    b.BusModelID,
    bm.BusModel,
    bm.ModelQuality,
    DATEDIFF(YEAR, b.DateFirstInService, li.LineItemDate) AS NumberOfYearsInService,
    ec.ExpenseCategoryID,
    ec.ExpenseCategory,
    v.VendorID,
    v.VendorName,
    e.EmployeeID,
    CONCAT(e.FirstName, ' ', e.LastName) AS EmployeeName,
    CONCAT(ce.FirstName, ' ', ce.LastName) AS CountersignerEmployeeName,
    li.Amount
FROM dbo.LineItem li
    INNER JOIN dbo.Calendar c
        ON li.LineItemDate = c.Date
    INNER JOIN dbo.Bus b
        ON li.BusID = b.BusID
    INNER JOIN dbo.BusModel bm
        ON b.BusModelID = bm.BusModelID
    INNER JOIN dbo.ExpenseCategory ec
        ON li.ExpenseCategoryID = ec.ExpenseCategoryID
    INNER JOIN dbo.Vendor v
        ON li.VendorID = v.VendorID
    INNER JOIN dbo.Employee e
        ON li.EmployeeID = e.EmployeeID
    LEFT OUTER JOIN dbo.Employee ce
        ON li.CountersignerEmployeeID = ce.EmployeeID;""", conn)
line_items['FirstDayOfMonth'] = pd.to_datetime(line_items['FirstDayOfMonth'])

## A Gentle Reminder of the Issue

We want to investigate the dramatic uptick in invoices for the years 2019-2021 to figure out if this is normal behavior or if it is indicative of an underlying issue, up to and including fraud.

In [ ]:
import seaborn
import matplotlib.pyplot as plt

items_by_year = line_items.groupby("CalendarYear").count()
ax = seaborn.lineplot(data=items_by_year, x="CalendarYear", y="LineItemID")
ax.set(xlabel='Calendar Year', ylabel='Line Items', title='Line Items per Calendar Year')
plt.show()

## Breakdowns by Time

One way that we can slice cohorts of data is by time.  For example, we can look at monthly invoice counts on a year-by-year basis to gain an understanding of behavior.  We can use a box plot to visualize counts by month for each year.

In [ ]:
import datetime

items_by_month_and_year = line_items[["FirstDayOfMonth", "LineItemID"]].groupby(["FirstDayOfMonth"]).count()
items_by_month_and_year["CalendarYear"] = items_by_month_and_year.index.year
items_by_month_and_year["FirstDayOfMonth"] = items_by_month_and_year.index
items_by_month_and_year

plt.figure(figsize=(18,8))
ax = seaborn.boxplot(data=items_by_month_and_year, x="CalendarYear", y="LineItemID")
ax.set(xlabel='Calendar Year', ylabel='Line Items', title='Box Plot of Monthly Line Items per Year')
plt.show()

2019-2021 has a **huge** spread compared to the other years.  What's interesting about this is that it seems there might be at least one month in 2019 and at lest one month in 2021 which fit the old pattern--we can see the bottom tails of the box plot fall somewhat in line with the normal trend of 2011-2018 and 2022.

Let's lay out month on a scatter plot to make this even clearer.

In [ ]:
plt.figure(figsize=(18,8))
seaborn.scatterplot(data=items_by_month_and_year, x="FirstDayOfMonth", y="LineItemID")
ax.set(xlabel='Calendar Month', ylabel='Line Items', title='Scatter Plot of Line Items per Month')
plt.show()

Something we can do with Seaborn to make understanding the year splits better is to set the hue by year.

In [ ]:
plt.figure(figsize=(18,8))
seaborn.scatterplot(data=items_by_month_and_year, x="FirstDayOfMonth", y="LineItemID", hue="CalendarYear", palette="colorblind")
ax.set(xlabel='Calendar Month', ylabel='Line Items', title='Scatter Plot of Line Items per Month')
plt.show()

Adding in coloration here, we can see that 2 or perhaps 3 months in 2019 fit with the existing pattern, as well as 3 months in 2021.  The remaining 9-10 months of 2019, all of 2020, and 9 months of 2021 fit a new pattern.  Let's look specifically at 2019:

In [ ]:
items_by_month_and_year.query("CalendarYear == 2019")

January and February are the two low months, with March being the hard-to-discern month. The rest of the year is considerably higher. Now let's look at 2021:

In [ ]:
items_by_month_and_year.query("CalendarYear == 2021")

All of the high months start early in the year and drop down to normal numbers in October. This is consistent with a behavioral change from March (or April) 2019 through September 2021.

### Breakdowns by Expense Category

Let's move over to expense categories and see what these look like.

In [ ]:
expense_categories_by_month = line_items[["ExpenseCategory", "FirstDayOfMonth", "LineItemID", "Amount"]].groupby(
    ["ExpenseCategory", "FirstDayOfMonth"], as_index=False).agg({"LineItemID":"count", "Amount":"sum"})

plt.figure(figsize=(12,14))
ax = seaborn.boxplot(data=expense_categories_by_month, x="LineItemID", y="ExpenseCategory")
ax.set(xlabel='Line Items', ylabel='Expense Category', title='Box Plot of Monthly Line Items per Expense Category')
plt.show()

We might also be interested in looking at the cost of each individual invoice by expense category. I'm cutting off the chart at 10 USD because, due to the way I generated data, there are a fair number of invoices with unrealistically low amounts.

In [ ]:
plt.figure(figsize=(12,14))
ax = seaborn.boxplot(data=line_items.query("Amount > 10"), x="Amount", y="ExpenseCategory")
ax.set(xscale="log", xlabel='Amount Spent', ylabel='Expense Category', title='Box Plot of Amount Spent per Expense Category')
plt.show()

Going back to the prior visual, it appeared that there were four expense categories with a large number of outlier months in terms of numbers of invoices: Glass, Lights, Mirrors, and Windshield Wipers. Let's take a look at each of those individually and see what we can learn.

In [ ]:
import sparklines as spk
import numpy as np

def sparkline_str(x):
    bins = np.histogram(x)[0]
    sl = ''.join(spk.sparklines(bins))
    return sl
sparkline_str.__name__ = "sparkline"

In [ ]:
vendor_expenses_by_month = line_items[["VendorName", "ExpenseCategory", "FirstDayOfMonth", "LineItemID", "Amount"]].query(
    "ExpenseCategory.isin(['Glass, Windshields & Windows', 'Lights & Reflectors', 'Mirrors', 'Windshield Wipers & Parts'])").groupby(
    ["VendorName", "ExpenseCategory", "FirstDayOfMonth"], as_index=False).agg({"LineItemID":"count", "Amount":"sum"})

format_dict = {'sum': '${0:,.0f}', 'mean': '${0:,.2f}'}
vendor_expenses_by_month \
    .groupby(["ExpenseCategory", "VendorName"])['Amount'] \
    .agg(['mean', 'sum', sparkline_str]) \
    .style.format(format_dict)


Note that the behavior of Glass and Sons differs markedly from all other vendors within these four categories. Building this out as a box plot will make it even plainer.

In [ ]:
import matplotlib.ticker as mtick
plt.figure(figsize=(8,8))

for expense_category in vendor_expenses_by_month["ExpenseCategory"].drop_duplicates():
    ax = seaborn.boxplot(data=vendor_expenses_by_month.where(
        vendor_expenses_by_month["ExpenseCategory"]==expense_category), x="Amount", y="VendorName")
    ax.set(xscale="log", xlabel='Amount Spent', ylabel='Vendor Name', title=expense_category)
    plt.show()

Glass and Sons is the only vendor which exhibits this strange behavior of a large number of positive outliers. Remember that each outlier represents a separate month, so this is a multi-year period. If there was a systemic issue around supplies, we would expect the other vendors to behave similarly during this stretch. Also, not surprisingly, those outlier months are in the 2019-2021 range.

Now let's look at the number of invoices per month handled by each vendor. We'll do this as a faceted line chart over time.

In [ ]:
plt.figure(figsize=(8,8))
g = seaborn.FacetGrid(vendor_expenses_by_month, row="ExpenseCategory", col="VendorName", aspect=1.3, margin_titles=True)
g.map_dataframe(seaborn.lineplot, x="FirstDayOfMonth", y="LineItemID")
g.set_axis_labels("Calendar Month", "Number of Line Items")
g.set_titles(col_template="{col_name}", row_template="{row_name}")
g.tight_layout()

What we can see is that Glass and Sons experiences a precipitous jump during 2019 across all of its expense categories, whereas its competitors see no such movement. We can also look at the average cost per line item for more information:

In [ ]:
mean_vendor_expenses_by_month = line_items[["VendorName", "ExpenseCategory", "FirstDayOfMonth", "LineItemID", "Amount"]].query(
    "ExpenseCategory.isin(['Glass, Windshields & Windows', 'Lights & Reflectors', 'Mirrors', 'Windshield Wipers & Parts'])").groupby(
    ["VendorName", "ExpenseCategory", "FirstDayOfMonth"], as_index=False).agg({"LineItemID":"count", "Amount":"mean"})

g = seaborn.FacetGrid(mean_vendor_expenses_by_month, row="ExpenseCategory", col="VendorName", aspect=1.3, margin_titles=True)
g.map_dataframe(seaborn.lineplot, x="FirstDayOfMonth", y="Amount")
g.set_axis_labels("Calendar Month", "Mean Cost (USD)")
g.set_titles(col_template="{col_name}", row_template="{row_name}")
g.tight_layout()

During the stretch in which Glass and Sons has so many invoices, it is also consistently more expensive than competitors. The increase in lights and reflectors is not quite as pronounced, but for the others, it is obvious that there is something funny going on. It's not normal behavior to see the price of one competitor go up so much and get **more** buisness over a significant stretch, especially when there are other firms in most of these expense categories.  What we'd normally expect to see is that if average price goes up considerably, there would be a shift in demand to other vendors who offer a better deal.

## Analyzing Employees

We don't have proof of foul play, but we do have the start:  abnormal behavior leading to major increases in expenditures.  And it's not like market circumstances have driven the changes:  the other players in the market are still doing their thing and charging about the same price they historically have.  If there is fraud, we would expect to see at least one person on the inside behaving differently.  This is where cohort analysis really shines.  First, we'll crease a box plot to look at the dollar amounts different employees handle in their invoices.

In [ ]:
plt.figure(figsize=(8,8))

ax = seaborn.boxplot(data=line_items.query("Amount > 10"), x="Amount", y="EmployeeName")
ax.set(xscale="log", xlabel='Amount Spent', ylabel='Employee', title='Box Plot of Amount Spent per Employee')
plt.show()

On the whole, these are quite similar. Everybody has a pretty similar mean and a similar spread of outliers. Let's now focus on Glass and Sons invoices, splitting between pre-2019 and post-2019. First, the pre-2019 data:

In [ ]:
plt.figure(figsize=(8,8))
ax = seaborn.boxplot(
    data=line_items.query("Amount > 10 & VendorName.str.startswith('Glass and Sons') & CalendarYear < 2019"), x="Amount", y="EmployeeName")
ax.set(xlabel='Amount Spent', ylabel='Employee', title='Box Plot of Amount Spent per Employee')
plt.show()

Again, everybody's behavior is approximately the same. Now let's look at the post-2019 behavior.

In [ ]:
plt.figure(figsize=(8,8))
ax = seaborn.boxplot(
    data=line_items.query("Amount > 10 & VendorName.str.startswith('Glass and Sons') & CalendarYear >= 2019"), x="Amount", y="EmployeeName")
ax.set(xlabel='Amount Spent', ylabel='Employee', title='Box Plot of Amount Spent per Employee')
plt.show()

At this point, we see the combination of Villiers, Sweeting, Mowett, and Harte pop up. Note that even though we have included at least 16 months which we don't think contain suspicious data, their averages are still well above everybody else, to the point where their interquartile ranges span beyond everybody else's biggest outliers. Nobody in the group of eight had an amount spent over ~620 USD or so, but our four outlier-drivers had numbers over 1000 USD. What we'll do now is take a look at the line plot of expenditures by month per employee to drive this home.

In [ ]:
plt.figure(figsize=(14,8))
employee_amounts = line_items[["VendorName", "EmployeeName", "CalendarYear", "FirstDayOfMonth", "LineItemID", "Amount"]] \
    .query("Amount > 10 & VendorName.str.startswith('Glass and Sons')") \
    .groupby(["EmployeeName", "FirstDayOfMonth"], as_index=False).agg({"LineItemID":"count", "Amount":"sum"})

ax = seaborn.lineplot(data=employee_amounts, y="Amount", x="FirstDayOfMonth", hue="EmployeeName")
ax.set(xlabel='Amount Spent', ylabel='Expenditures (USD)', title='Monthly Expenditures to Glass and Sons by Employee')
plt.show()

These are now persons of interest and we are starting to get evidence of a plot. There may still be a valid explanation for this, but it is a bit harder to believe when 8 of the 12 employees handling invoices didn't change their behaviors over the same time frame. And here's what it looks like for everybody **not** named Glass and Sons.

In [ ]:
plt.figure(figsize=(14,8))
non_glass_employee_amounts = line_items[["VendorName", "EmployeeName", "CalendarYear", "FirstDayOfMonth", "LineItemID", "Amount"]] \
    .query("Amount > 10 & not VendorName.str.startswith('Glass and Sons')") \
    .groupby(["EmployeeName", "FirstDayOfMonth"], as_index=False).agg({"LineItemID":"count", "Amount":"sum"})

ax = seaborn.lineplot(data=non_glass_employee_amounts, y="Amount", x="FirstDayOfMonth", hue="EmployeeName")
ax.set(xlabel='Amount Spent', ylabel='Expenditures (USD)', title='Monthly Expenditures to Venders Other Than Glass and Sons by Employee')
plt.show()

We see consistent behavior over all of the other vendors, even for our four employees of interest.

## Clustering Behavior

One last area of investigation is clustering behavior.  We want to see if there are clusters of invoice amounts in our data set.  Knowing that 1000 USD is the point where we need two signatures, we want to see if there is any special behavior near that point.

In [ ]:
plt.figure(figsize=(14,8))
seaborn.histplot(data=line_items, x="Amount", bins=100)

There is a downward curve and maybe a little bit of a bump at 1000 USD, though it looks like the bump might be **after** 1000 USD.  Let's now separate out Glass and Sons from the rest of the set to see how they look in comparison.  First, everybody else:

In [ ]:
plt.figure(figsize=(14,8))
seaborn.histplot(data=line_items.query("not VendorName.str.startswith('Glass and Sons')"), x="Amount", bins=100)

Let's zoom in on data between 750 USD and 1500 USD.

In [ ]:
plt.figure(figsize=(14,8))
seaborn.histplot(data=line_items.query("not VendorName.str.startswith('Glass and Sons') & Amount >= 750 & Amount < 1500"), x="Amount", bins=100)

Looking at this range, there doesn't appear to be any clustering behavior among vendors trying to stay under 1000 USD. There's local spikiness, but we can see a definite trend here.

And now here's Glass and Sons:

In [ ]:
plt.figure(figsize=(14,8))
seaborn.histplot(data=line_items.query("VendorName.str.startswith('Glass and Sons')"), x="Amount", bins=100)

The two peaks is a little strange, but it's something we can explain: they sell in different categories with different price points, so there won't necessarily be one smooth curve.

But note that there is a small but definite bump at 1000 USD. This is what we saw earlier with a large number of records between 999 USD and 999.99 USD. So let's take a closer look at just the 750 to 1500 range for Glass and Sons.

In [ ]:
plt.figure(figsize=(14,8))
seaborn.histplot(data=line_items.query("VendorName.str.startswith('Glass and Sons') & Amount >= 750 & Amount < 1500"), x="Amount", bins=100)

This makes it look obvious that they're trying to stay under 1000 USD, with just a few over that 1000 mark. Because we know there is a counter-signer at 1000 USD, let's take a look at who signed and counter-signed for these dual-signer line items, as well as when they happened.

In [ ]:
line_items.query("VendorName.str.startswith('Glass and Sons') & Amount >= 1000")[["FirstDayOfMonth", "Amount", "EmployeeName", "CountersignerEmployeeName"]]

For reference, the names of interest are Villiers, Sweeting, Mowett, and Harte. And for every one of these invoices, we see one of those four names for the `EmployeeName` column **as well as** another of these four for the `CountersignerEmployeeName` column.  These four people alone have dealt with all of the high-expense line items for Glass and Sons.  It might be possible that these are the only four allowed to counter-sign, so let's check that before we go too far:

In [ ]:
line_items[["CountersignerEmployeeName", "LineItemID"]].groupby("CountersignerEmployeeName").count()

Nope. It turns out that everybody counter-signs at about the same rate.